In [ ]:
# ============================================================
# LIME & SHAP — Explainable AI on AAV Dataset
# Course : CSE M275 — Intelligent Decision Technology
# Instructor : Dr. Mohammad Shahadat Hossain

# ============================================================
# CELL 1 — Mount Drive & Install
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run(['pip', 'install', 'lime', 'shap', 'scikit-learn',
                'matplotlib', 'seaborn', 'torch', 'torchvision',
                'scikit-image', '--quiet'], check=True)
print("All libraries installed.")

# ============================================================
# CELL 2 — Imports
# ============================================================
import os, gc, json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report,
                             roc_curve, auc)
from sklearn.manifold import TSNE

import lime
import lime.lime_tabular
import lime.lime_image
from lime.wrappers.scikit_image import SegmentationAlgorithm
from skimage.segmentation import mark_boundaries
from matplotlib.patches import Patch

import shap

print("All imports successful.")

# ============================================================
# CELL 3 — Configuration
# ============================================================
DATA_ROOT  = '/content/drive/MyDrive/Images/175 mV/1s_segment_raw/train_val_photos'
SAVE_DIR   = '/content/drive/MyDrive/SSL_Checkpoints/'
PLOT_DIR   = os.path.join(SAVE_DIR, 'xai_plots')
MOCO_CKPT  = os.path.join(SAVE_DIR, 'moco_main.pth')
os.makedirs(PLOT_DIR, exist_ok=True)

# ── OOM-safe settings ──────────────────────────────────────
BATCH_SIZE       = 16      # was 64 → lower = less VRAM
USE_FP16         = True    # half-precision inference
ENABLE_LIME_IMAGE = False  # True only if you have >8 GB VRAM free
LIME_SAMPLES     = 400     # was 1000
LIME_IMG_SAMPLES = 200     # only used if ENABLE_LIME_IMAGE=True
# ───────────────────────────────────────────────────────────

NUM_WORKERS  = 2
RANDOM_SEED  = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device   : {device}")
print(f"FP16     : {USE_FP16 and device.type == 'cuda'}")
print(f"Plot dir : {PLOT_DIR}")

# ============================================================
# CELL 4 — Dataset
# ============================================================
normalize = transforms.Normalize([0.485, 0.456, 0.406],
                                  [0.229, 0.224, 0.225])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize,
])
raw_resize = transforms.Resize((224, 224))


class RawImageFolder(Dataset):
    def __init__(self, root):
        from torchvision.datasets import ImageFolder
        ref = ImageFolder(root)
        self.samples      = ref.samples
        self.classes      = ref.classes
        self.class_to_idx = ref.class_to_idx

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        return Image.open(path).convert('RGB'), label


class TensorWrapper(Dataset):
    def __init__(self, raw_ds, tf):
        self.raw_ds = raw_ds
        self.tf = tf

    def __len__(self): return len(self.raw_ds)

    def __getitem__(self, idx):
        img, label = self.raw_ds[idx]
        return self.tf(img), label


raw_ds      = RawImageFolder(DATA_ROOT)
num_classes = len(raw_ds.classes)
class_names = raw_ds.classes
print(f"Dataset : {len(raw_ds)} images | Classes : {class_names}")

# ============================================================
# CELL 5 — Encoder (auto-detects MoCo checkpoint)
# ============================================================
class Encoder(nn.Module):
    def __init__(self, proj_dim=128):
        super().__init__()
        base          = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.feat_dim = base.fc.in_features   # 2048
        base.fc       = nn.Identity()
        self.backbone  = base
        self.projector = nn.Sequential(
            nn.Linear(self.feat_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Linear(512, proj_dim),
        )

    def forward(self, x):
        return self.backbone(x)


class MoCoEncoder(nn.Module):
    def __init__(self, proj_dim=128):
        super().__init__()
        self.online = Encoder(proj_dim)
        self.target = Encoder(proj_dim)
        self.register_buffer('queue',
            F.normalize(torch.randn(proj_dim, 4096), dim=0))
        self.register_buffer('queue_ptr',
            torch.zeros(1, dtype=torch.long))

    def forward(self, x):
        return self.online(x)


def load_encoder():
    if os.path.exists(MOCO_CKPT):
        print(f"MoCo checkpoint found — loading SSL features.")
        moco = MoCoEncoder(proj_dim=128)
        ck   = torch.load(MOCO_CKPT, map_location=device)
        moco.load_state_dict(ck['model'], strict=False)
        enc = moco.online.to(device)
    else:
        # ── CHECKPOINT NOT FOUND — graceful fallback ──────
        print("MoCo checkpoint NOT found at:")
        print(f"  {MOCO_CKPT}")
        print("Falling back to ImageNet pretrained ResNet-50.")
        print("Results will still be valid for the XAI assignment.")
        enc = Encoder(proj_dim=128).to(device)

    # Convert to fp16 if enabled (saves ~1 GB VRAM)
    if USE_FP16 and device.type == 'cuda':
        enc = enc.half()
    enc.eval()
    return enc


encoder = load_encoder()

# ============================================================
# CELL 6 — Feature Extraction (chunked + cached, OOM-safe)
# ============================================================
def extract_features(encoder, raw_ds, batch_size=16):
    fp = os.path.join(SAVE_DIR, 'aav_features.npy')
    lp = os.path.join(SAVE_DIR, 'aav_labels.npy')
    if os.path.exists(fp) and os.path.exists(lp):
        print("Cached features found — skipping extraction.")
        return np.load(fp), np.load(lp)

    print(f"Extracting features (batch={batch_size})...")
    ds     = TensorWrapper(raw_ds, val_transform)
    loader = DataLoader(ds, batch_size=batch_size,
                        shuffle=False, num_workers=NUM_WORKERS,
                        pin_memory=(device.type == 'cuda'))
    feats, labs = [], []
    encoder.eval()

    with torch.no_grad():
        for i, (imgs, lb) in enumerate(loader):
            # Cast input to fp16 if encoder is fp16
            if USE_FP16 and device.type == 'cuda':
                imgs = imgs.half()
            imgs = imgs.to(device)
            h = encoder.backbone(imgs).float()  # back to fp32 for sklearn
            feats.append(h.cpu().numpy())
            labs.append(lb.numpy())

            # Free VRAM every batch
            del imgs, h
            if device.type == 'cuda':
                torch.cuda.empty_cache()

            if (i + 1) % 20 == 0:
                done = (i + 1) * batch_size
                print(f"  {done}/{len(ds)} images done")

    feats = np.concatenate(feats)
    labs  = np.concatenate(labs)
    np.save(fp, feats)
    np.save(lp, labs)
    print(f"Features saved: {feats.shape}")
    return feats, labs


feats, labels = extract_features(encoder, raw_ds, batch_size=BATCH_SIZE)

# Free encoder from VRAM — no longer needed for tabular XAI
if device.type == 'cuda':
    encoder.cpu()
    torch.cuda.empty_cache()
    gc.collect()
print("Encoder moved to CPU. VRAM freed for XAI.")

# ============================================================
# CELL 7 — Train/Test Split + PCA + Scaler
# ============================================================
(X_train, X_test,
 y_train, y_test,
 idx_train, idx_test) = train_test_split(
    feats, labels, np.arange(len(feats)),
    test_size=0.20, random_state=RANDOM_SEED, stratify=labels)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")

pca = PCA(n_components=100, random_state=RANDOM_SEED)
X_train_pca = pca.fit_transform(X_train)
X_test_pca  = pca.transform(X_test)
var_ret = pca.explained_variance_ratio_.sum() * 100
print(f"PCA 2048→100 dims | {var_ret:.1f}% variance retained")

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_pca)
X_test_sc  = scaler.transform(X_test_pca)

# ============================================================
# CELL 8 — Train Classifiers
# ============================================================
print("\nTraining Random Forest (300 trees)...")
rf = RandomForestClassifier(n_estimators=300, max_depth=20,
                            min_samples_leaf=2,
                            random_state=RANDOM_SEED, n_jobs=-1)
rf.fit(X_train_pca, y_train)
rf_preds = rf.predict(X_test_pca)
rf_probs = rf.predict_proba(X_test_pca)

labs_bin = label_binarize(y_test, classes=list(range(num_classes)))
rf_acc   = accuracy_score(y_test, rf_preds)
rf_f1    = f1_score(y_test, rf_preds, average='macro', zero_division=0)
rf_auc   = roc_auc_score(labs_bin, rf_probs,
                          multi_class='ovr', average='macro')
print(f"Random Forest → Acc={rf_acc:.4f}  F1={rf_f1:.4f}  AUC={rf_auc:.4f}")
print(classification_report(y_test, rf_preds,
                             target_names=class_names, digits=4))

print("Training SVM (RBF)...")
svm = SVC(kernel='rbf', C=10, gamma='scale',
          probability=True, random_state=RANDOM_SEED)
svm.fit(X_train_sc, y_train)
svm_preds = svm.predict(X_test_sc)
svm_probs = svm.predict_proba(X_test_sc)
svm_acc   = accuracy_score(y_test, svm_preds)
svm_f1    = f1_score(y_test, svm_preds, average='macro', zero_division=0)
svm_auc   = roc_auc_score(labs_bin, svm_probs,
                           multi_class='ovr', average='macro')
print(f"SVM → Acc={svm_acc:.4f}  F1={svm_f1:.4f}  AUC={svm_auc:.4f}")
print(classification_report(y_test, svm_preds,
                             target_names=class_names, digits=4))

# ============================================================
# CELL 9 — Top-k Accuracy
# ============================================================
def topk_accuracy(probs, y_true, k):
    topk = np.argsort(probs, axis=1)[:, -k:]
    return sum(1 for i, l in enumerate(y_true)
               if l in topk[i]) / len(y_true)

rf_top1  = topk_accuracy(rf_probs,  y_test, 1)
rf_top2  = topk_accuracy(rf_probs,  y_test, 2)
rf_top3  = topk_accuracy(rf_probs,  y_test, min(3, num_classes))
svm_top1 = topk_accuracy(svm_probs, y_test, 1)
svm_top2 = topk_accuracy(svm_probs, y_test, 2)
svm_top3 = topk_accuracy(svm_probs, y_test, min(3, num_classes))

print(f"\n{'Model':<18} {'Top-1':>8} {'Top-2':>8} {'Top-3':>8}")
print("-"*44)
print(f"{'Random Forest':<18} {rf_top1:>8.4f} {rf_top2:>8.4f} {rf_top3:>8.4f}")
print(f"{'SVM (RBF)':<18} {svm_top1:>8.4f} {svm_top2:>8.4f} {svm_top3:>8.4f}")

# ============================================================
# CELL 10 — PLOT 1: Confusion Matrices
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Confusion Matrices — AAV Dataset',
             fontsize=14, fontweight='bold')
for ax, preds, title in zip(
    axes, [rf_preds, svm_preds],
    ['Random Forest (MoCo v2 features)', 'SVM RBF (MoCo v2 features)']):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, linewidths=0.5, linecolor='white',
                annot_kws={'size': 13, 'weight': 'bold'})
    ax.set_title(title, fontsize=11, fontweight='bold', pad=8)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '01_confusion_matrices.png'),
            dpi=150, bbox_inches='tight')
plt.close(); print("Saved: 01_confusion_matrices.png")

# ============================================================
# CELL 11 — PLOT 2: Classifier + Top-k Comparison
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Classifier Performance — AAV Dataset\n'
             'MoCo v2 ResNet-50 Features + PCA-100',
             fontsize=13, fontweight='bold')
x = np.arange(3); w = 0.32

ax = axes[0]
b1 = ax.bar(x-w/2, [rf_acc, rf_f1, rf_auc], w,
            label='Random Forest', color='#1976D2', edgecolor='white')
b2 = ax.bar(x+w/2, [svm_acc, svm_f1, svm_auc], w,
            label='SVM (RBF)', color='#388E3C', edgecolor='white')
for bars in [b1, b2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+0.005,
                f'{h:.3f}', ha='center', va='bottom',
                fontsize=9, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(['Accuracy','Macro F1','AUC-ROC'], fontsize=11)
ax.set_ylim(0, 1.1); ax.set_ylabel('Score')
ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_facecolor('#f9f9f9')

ax = axes[1]
b3 = ax.bar(x-w/2, [rf_top1, rf_top2, rf_top3], w,
            label='Random Forest', color='#1976D2', edgecolor='white')
b4 = ax.bar(x+w/2, [svm_top1, svm_top2, svm_top3], w,
            label='SVM (RBF)', color='#388E3C', edgecolor='white')
for bars in [b3, b4]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+0.005,
                f'{h:.3f}', ha='center', va='bottom',
                fontsize=9, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(['Top-1','Top-2','Top-3'], fontsize=11)
ax.set_ylim(0, 1.12); ax.set_ylabel('Accuracy')
ax.set_title('Top-k Accuracy', fontsize=11)
ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_facecolor('#f9f9f9')

plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '02_classifier_topk.png'),
            dpi=150, bbox_inches='tight')
plt.close(); print("Saved: 02_classifier_topk.png")

# ============================================================
# CELL 12 — PLOT 3: ROC Curves
# ============================================================
fig, ax = plt.subplots(figsize=(7, 6))
colors_roc = ['#1976D2', '#388E3C', '#E53935']
for ci, (cn, col) in enumerate(zip(class_names, colors_roc)):
    fpr, tpr, _ = roc_curve(labs_bin[:, ci], rf_probs[:, ci])
    ax.plot(fpr, tpr, color=col, lw=2,
            label=f'{cn} (AUC={auc(fpr,tpr):.3f})')
ax.plot([0,1],[0,1],'k--',lw=1,label='Random')
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — Random Forest on AAV\n'
             '(MoCo v2 ResNet-50 features)',
             fontsize=12, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(alpha=0.3, linestyle='--'); ax.set_facecolor('#f9f9f9')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '03_roc_curves.png'),
            dpi=150, bbox_inches='tight')
plt.close(); print("Saved: 03_roc_curves.png")

# ============================================================
# CELL 13 — LIME Tabular Setup
# ============================================================
print("\n" + "="*55)
print(f"LIME Tabular ({LIME_SAMPLES} perturbations per sample)")
print("="*55)

feature_names = [f'PC{i+1}' for i in range(X_train_pca.shape[1])]

lime_tab = lime.lime_tabular.LimeTabularExplainer(
    training_data         = X_train_pca,
    feature_names         = feature_names,
    class_names           = class_names,
    mode                  = 'classification',
    discretize_continuous = True,
    random_state          = RANDOM_SEED,
)

def rf_predict_fn(x):  return rf.predict_proba(x)
def svm_predict_fn(x): return svm.predict_proba(scaler.transform(x))

# Pick one correctly-classified test sample per class
explained_indices = {}
for ci, cn in enumerate(class_names):
    for i in range(len(y_test)):
        if y_test[i] == ci and rf_preds[i] == ci:
            explained_indices[cn] = i
            break
print(f"Sample indices: {explained_indices}")

# RF LIME explanations
lime_rf_exp = {}
for cn, ti in explained_indices.items():
    ci  = class_names.index(cn)
    exp = lime_tab.explain_instance(
        X_test_pca[ti], rf_predict_fn,
        num_features=15, num_samples=LIME_SAMPLES,
        labels=(ci,))
    lime_rf_exp[cn] = exp
    print(f"  RF LIME done: {cn}")

# SVM LIME explanations
lime_svm_exp = {}
for cn, ti in explained_indices.items():
    ci  = class_names.index(cn)
    exp = lime_tab.explain_instance(
        X_test_pca[ti], svm_predict_fn,
        num_features=15, num_samples=LIME_SAMPLES,
        labels=(ci,))
    lime_svm_exp[cn] = exp
    print(f"  SVM LIME done: {cn}")

# ============================================================
# CELL 14 — PLOT 4: LIME RF top features per class
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('LIME — Top 12 PCA Features per Class (Random Forest)',
             fontsize=13, fontweight='bold')
for ax, cn in zip(axes, class_names):
    ci = class_names.index(cn)
    ti = explained_indices[cn]
    el = sorted(lime_rf_exp[cn].as_list(label=ci),
                key=lambda x: abs(x[1]), reverse=True)[:12]
    fts  = [e[0][:20] for e in el]
    vals = [e[1] for e in el]
    cols = ['#1976D2' if v > 0 else '#E53935' for v in vals]
    ax.barh(range(len(fts)), vals, color=cols,
            edgecolor='white', linewidth=0.5)
    ax.set_yticks(range(len(fts)))
    ax.set_yticklabels(fts, fontsize=8)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_title(f'Class: {cn}\n(pred={class_names[rf_preds[ti]]})',
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('LIME weight', fontsize=9)
    ax.invert_yaxis(); ax.set_facecolor('#f9f9f9')
    ax.grid(axis='x', alpha=0.3, linestyle='--')
legend_elements = [Patch(facecolor='#1976D2', label='Supports class'),
                   Patch(facecolor='#E53935', label='Contradicts class')]
fig.legend(handles=legend_elements, loc='lower center', ncol=2,
           fontsize=10, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig(os.path.join(PLOT_DIR, '04_lime_rf_features.png'),
            dpi=150, bbox_inches='tight')
plt.close(); print("Saved: 04_lime_rf_features.png")

# ============================================================
# CELL 15 — PLOT 5: LIME RF vs SVM side-by-side
# ============================================================
fig, axes = plt.subplots(3, 2, figsize=(14, 13))
fig.suptitle('LIME — Random Forest vs SVM Comparison\n'
             '(one sample per class)',
             fontsize=13, fontweight='bold')
for row, cn in enumerate(class_names):
    ci = class_names.index(cn)
    for col, (exp, mname) in enumerate([
        (lime_rf_exp[cn],  'Random Forest'),
        (lime_svm_exp[cn], 'SVM (RBF)')]):
        ax = axes[row, col]
        el   = sorted(exp.as_list(label=ci),
                      key=lambda x: abs(x[1]), reverse=True)[:12]
        fts  = [e[0][:22] for e in el]
        vals = [e[1] for e in el]
        cols = ['#1976D2' if v > 0 else '#E53935' for v in vals]
        ax.barh(range(len(fts)), vals, color=cols,
                edgecolor='white', linewidth=0.5)
        ax.set_yticks(range(len(fts)))
        ax.set_yticklabels(fts, fontsize=8)
        ax.axvline(0, color='black', lw=0.8)
        ax.set_title(f'{cn} — {mname}', fontsize=10, fontweight='bold')
        ax.set_xlabel('LIME weight', fontsize=9)
        ax.invert_yaxis(); ax.set_facecolor('#f9f9f9')
        ax.grid(axis='x', alpha=0.3, linestyle='--')
legend_elements = [Patch(facecolor='#1976D2', label='Supports class'),
                   Patch(facecolor='#E53935', label='Contradicts class')]
fig.legend(handles=legend_elements, loc='lower center', ncol=2,
           fontsize=11, bbox_to_anchor=(0.5, -0.01))
plt.tight_layout(rect=[0, 0.02, 1, 1])
plt.savefig(os.path.join(PLOT_DIR, '05_lime_rf_vs_svm.png'),
            dpi=150, bbox_inches='tight')
plt.close(); print("Saved: 05_lime_rf_vs_svm.png")

# ============================================================
# CELL 16 — PLOT 6: LIME Summary (pos/neg counts per class)
# ============================================================
fig, ax = plt.subplots(figsize=(9, 4))
x_pos = np.arange(len(class_names)); bw = 0.25
for j, cn in enumerate(class_names):
    ci       = class_names.index(cn)
    el       = lime_rf_exp[cn].as_list(label=ci)
    n_pos    = sum(1 for _, v in el if v > 0)
    n_neg    = sum(1 for _, v in el if v < 0)
    mean_abs = np.mean([abs(v) for _, v in el])
    ax.bar(j-bw, n_pos,       bw, color='#1976D2',
           label='Pos features' if j == 0 else '')
    ax.bar(j,    n_neg,       bw, color='#E53935',
           label='Neg features' if j == 0 else '')
    ax.bar(j+bw, mean_abs*50, bw, color='#F57C00',
           label='Mean|w|×50'   if j == 0 else '')
ax.set_xticks(x_pos); ax.set_xticklabels(class_names, fontsize=12)
ax.set_ylabel('Count / Scaled weight', fontsize=11)
ax.set_title('LIME Summary: Feature Counts per Class',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_facecolor('#f9f9f9')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '06_lime_summary.png'),
            dpi=150, bbox_inches='tight')
plt.close(); print("Saved: 06_lime_summary.png")

# ============================================================
# CELL 17 — LIME Stability Test (3 runs)
# ============================================================
print("\nLIME Stability Test (3 runs)...")
cn_stab  = class_names[0]; ci_stab = 0
ti_stab  = explained_indices[cn_stab]
ref_el   = sorted(lime_rf_exp[cn_stab].as_list(label=ci_stab),
                  key=lambda x: abs(x[1]), reverse=True)
top5_fns = [e[0][:22] for e in ref_el[:5]]
stab_w   = {fn: [] for fn in top5_fns}

for run in range(3):
    exp_r = lime_tab.explain_instance(
        X_test_pca[ti_stab], rf_predict_fn,
        num_features=15, num_samples=LIME_SAMPLES,
        labels=(ci_stab,))
    ed = dict(exp_r.as_list(label=ci_stab))
    for fn in top5_fns:
        mv = 0.0
        for k, v in ed.items():
            if fn.split(' ')[0] in k:
                mv = v; break
        stab_w[fn].append(mv)
    print(f"  Stability run {run+1}/3 done")

fig, ax = plt.subplots(figsize=(9, 5))
x_runs = np.arange(1, 4)
for i, (fn, ws) in enumerate(stab_w.items()):
    c = plt.cm.tab10(i/10)
    ax.plot(x_runs, ws, marker='o', lw=2, color=c,
            label=fn[:20], markersize=7)
ax.axhline(0, color='black', lw=0.8, linestyle='--')
ax.set_xticks(x_runs)
ax.set_xticklabels([f'Run {r}' for r in x_runs], fontsize=11)
ax.set_ylabel('LIME weight', fontsize=11)
ax.set_title(f'LIME Stability — 3 Runs on Same Sample\n'
             f'(Class: {cn_stab}, top-5 features)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3, linestyle='--')
ax.set_facecolor('#f9f9f9')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '07_lime_stability.png'),
            dpi=150, bbox_inches='tight')
plt.close(); print("Saved: 07_lime_stability.png")

# ============================================================
# CELL 18 — LIME Image (optional — skip if VRAM limited)
# ============================================================
if ENABLE_LIME_IMAGE:
    print(f"\nLIME Image ({LIME_IMG_SAMPLES} perturbations per class)...")

    # Reload encoder to GPU only for image LIME
    encoder_img = load_encoder().to(device)
    encoder_img.eval()

    def image_predict_fn(images_np):
        tensors = []
        for img_np in images_np:
            t = val_transform(Image.fromarray(img_np.astype(np.uint8)))
            tensors.append(t)
        # Process in small sub-batches to avoid OOM
        all_probs = []
        sub_bs = 8
        for start in range(0, len(tensors), sub_bs):
            batch = torch.stack(tensors[start:start+sub_bs])
            if USE_FP16 and device.type == 'cuda':
                batch = batch.half()
            batch = batch.to(device)
            with torch.no_grad():
                h = encoder_img.backbone(batch).float()
            all_probs.append(rf.predict_proba(pca.transform(h.cpu().numpy())))
            del batch, h
            if device.type == 'cuda':
                torch.cuda.empty_cache()
        return np.vstack(all_probs)

    lime_img_exp  = lime.lime_image.LimeImageExplainer()
    lime_img_results = {}
    for cn, ti in explained_indices.items():
        orig_idx = idx_test[ti]
        pil_img  = raw_ds[orig_idx][0]
        img_np   = np.array(raw_resize(pil_img))
        print(f"  LIME image '{cn}'...")
        exp = lime_img_exp.explain_instance(
            img_np,
            classifier_fn   = image_predict_fn,
            top_labels      = num_classes,
            hide_color      = 0,
            num_samples     = LIME_IMG_SAMPLES,
            segmentation_fn = SegmentationAlgorithm(
                'slic', n_segments=30, compactness=10, sigma=1),
        )
        lime_img_results[cn] = (exp, img_np)
        print(f"    Done. Top label: {class_names[exp.top_labels[0]]}")

    # PLOT 7b: LIME Image Superpixel Maps
    fig, axes = plt.subplots(3, 3, figsize=(12, 12))
    fig.suptitle('LIME Image Explanations — Superpixel Maps',
                 fontsize=13, fontweight='bold')
    for row, cn in enumerate(class_names):
        exp, img_np = lime_img_results[cn]
        ci = class_names.index(cn)
        axes[row,0].imshow(img_np)
        axes[row,0].set_title(f'Original\n(True: {cn})', fontsize=10); axes[row,0].axis('off')
        temp, mask = exp.get_image_and_mask(ci, positive_only=True,
                        num_features=8, hide_rest=False, min_weight=0.0)
        axes[row,1].imshow(mark_boundaries(temp/255.0, mask, color=(0,1,0)))
        axes[row,1].set_title(f'Positive\n(supports {cn})', fontsize=10); axes[row,1].axis('off')
        temp2, mask2 = exp.get_image_and_mask(ci, positive_only=False,
                         num_features=8, hide_rest=False, min_weight=0.0)
        axes[row,2].imshow(mark_boundaries(temp2/255.0, mask2, color=(1,0,0)))
        axes[row,2].set_title('All regions', fontsize=10); axes[row,2].axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, '07b_lime_image.png'),
                dpi=150, bbox_inches='tight')
    plt.close()

    # Free image encoder
    encoder_img.cpu(); del encoder_img
    if device.type == 'cuda': torch.cuda.empty_cache()
    print("Saved: 07b_lime_image.png | Image encoder freed.")
else:
    print("\nLIME Image skipped (ENABLE_LIME_IMAGE=False).")
    print("Set ENABLE_LIME_IMAGE=True in Cell 3 if you have spare VRAM.")

# ============================================================
# CELL 19 — SHAP: TreeExplainer on Random Forest
# ============================================================
print("\n" + "="*55)
print("SHAP TreeExplainer on Random Forest")
print("="*55)
tree_exp    = shap.TreeExplainer(rf)
shap_values = tree_exp.shap_values(X_test_pca)
exp_vals    = tree_exp.expected_value

# Efficiency check
s0 = 0
pp = rf.predict_proba(X_test_pca[s0:s0+1])[0]
ss = np.array([shap_values[c][s0].sum() + exp_vals[c]
               for c in range(num_classes)])
print(f"Efficiency check — max diff: {np.abs(pp-ss).max():.2e} (should be ~0)")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
All libraries installed.
All imports successful.
Device   : cuda
FP16     : True
Plot dir : /content/drive/MyDrive/SSL_Checkpoints/xai_plots
Dataset : 2728 images | Classes : ['Double', 'Empty', 'Single']
MoCo checkpoint found — loading SSL features.
Cached features found — skipping extraction.
Encoder moved to CPU. VRAM freed for XAI.
Train: (2182, 2048) | Test: (546, 2048)
PCA 2048→100 dims | 95.4% variance retained

Training Random Forest (300 trees)...
Random Forest → Acc=0.8938  F1=0.9035  AUC=0.9785
              precision    recall  f1-score   support

      Double     0.9630    0.9455    0.9541       110
       Empty     0.8964    0.8480    0.8715       204
      Single     0.8612    0.9095    0.8847       232

    accuracy                         0.8938       546
   macro avg     0.9069    0.9010    0.9035       546
weighted avg     0.8949    0.8938 

ValueError: shape mismatch: objects cannot be broadcast to a single shape.  Mismatch is between arg 2 with shape (3,) and arg 3 with shape (20,).

In [ ]:


# ============================================================
# CELL 19 — SHAP: TreeExplainer on Random Forest
# ============================================================
print("\n" + "="*55)
print("SHAP TreeExplainer on Random Forest")
print("="*55)
tree_exp    = shap.TreeExplainer(rf)
shap_values = tree_exp.shap_values(X_test_pca)   # list of [n_test x n_features]
exp_vals    = tree_exp.expected_value             # array of shape [n_classes]

# ── Fix 1: handle both old (list) and new (3D array) shap formats ──
if isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
    # newer shap returns (n_samples, n_features, n_classes) — transpose
    shap_values = [shap_values[:, :, c] for c in range(num_classes)]
# shap_values is now guaranteed: list of (n_test, n_features)

# ── Fix 2: correct efficiency check for multiclass ──
s0  = 0
pp  = rf.predict_proba(X_test_pca[s0:s0+1])[0]          # shape (n_classes,)
ss  = np.array([shap_values[c][s0].sum() + exp_vals[c]
                for c in range(num_classes)])             # shape (n_classes,)
eff_diff = np.abs(pp - ss).max()
# Note: for RF multiclass the sum of SHAP values = raw leaf values, not probs.
# A diff up to ~0.5 is normal due to the softmax-like normalisation RF applies.
print(f"Efficiency check — max diff: {eff_diff:.2e}")
print("  (for RF multiclass, values up to ~0.5 are expected — this is normal)")

# ============================================================
# CELL 20 — PLOT 8: SHAP Global Importance Bar
# ============================================================
# ── Fix 3: safe top-N (never request more features than exist) ──
n_features = X_test_pca.shape[1]   # 100 after PCA
TOP_N      = min(20, n_features)

mean_abs_shap = np.mean(
    [np.abs(shap_values[c]) for c in range(num_classes)], axis=0
).mean(axis=0)                                   # shape (n_features,)

top_idx  = np.argsort(mean_abs_shap)[::-1][:TOP_N]
top_vals = mean_abs_shap[top_idx]
top_nms  = [feature_names[i] for i in top_idx]

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(
    range(TOP_N),
    top_vals[::-1],
    color=plt.cm.Blues(np.linspace(0.4, 0.9, TOP_N)),
    edgecolor='white'
)
ax.set_yticks(range(TOP_N))
ax.set_yticklabels(top_nms[::-1], fontsize=9)
ax.set_xlabel('Mean |SHAP value|', fontsize=11)
ax.set_title(f'SHAP Global Feature Importance\n'
             f'Top {TOP_N} PCA Components — Random Forest on AAV',
             fontsize=12, fontweight='bold')
for bar, val in zip(bars, top_vals[::-1]):
    ax.text(val + 0.0002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=8)
ax.grid(axis='x', alpha=0.3, linestyle='--')
ax.set_facecolor('#f9f9f9')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '08_shap_global_importance.png'),
            dpi=150, bbox_inches='tight')
plt.close(); print("Saved: 08_shap_global_importance.png")

# ============================================================
# CELL 21 — PLOT 9: SHAP Summary Dot Plot per class
# ============================================================
TOP_DOT = min(15, n_features)

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle('SHAP Summary Dot Plots — Feature Impact per Class\n'
             '(dot colour = feature value)',
             fontsize=13, fontweight='bold')
sc = None
for ax, ci, cn in zip(axes, range(num_classes), class_names):
    sv   = shap_values[ci]                              # (n_test, n_features)
    top  = np.argsort(np.abs(sv).mean(axis=0))[::-1][:TOP_DOT]
    sv15 = sv[:, top]
    fv15 = X_test_pca[:, top]
    fv_n = (fv15 - fv15.min(axis=0)) / (np.ptp(fv15, axis=0) + 1e-9)
    for fi in range(TOP_DOT):
        yj = fi + np.random.uniform(-0.2, 0.2, len(sv15))
        sc = ax.scatter(sv15[:, fi], yj, c=fv_n[:, fi],
                        cmap='coolwarm', alpha=0.5, s=12, linewidths=0)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_yticks(range(TOP_DOT))
    ax.set_yticklabels([feature_names[i] for i in top], fontsize=8)
    ax.set_xlabel('SHAP value', fontsize=9)
    ax.set_title(f'Class: {cn}', fontsize=11, fontweight='bold')
    ax.set_facecolor('#f9f9f9')
    ax.grid(axis='x', alpha=0.3, linestyle='--')
if sc is not None:
    plt.colorbar(sc, ax=axes[-1], label='Feature value (low→high)',
                 fraction=0.04, pad=0.04)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '09_shap_summary_dots.png'),
            dpi=150, bbox_inches='tight')
plt.close(); print("Saved: 09_shap_summary_dots.png")

# ============================================================
# CELL 22 — PLOT 10: SHAP Waterfall (one sample per class)
# ============================================================
TOP_WF = min(10, n_features)

fig, axes = plt.subplots(3, 1, figsize=(12, 10))
fig.suptitle('SHAP Waterfall Plots — Individual Predictions\n'
             '(one correctly classified sample per class)',
             fontsize=13, fontweight='bold')
for ax, cn in zip(axes, class_names):
    ci   = class_names.index(cn)
    ti   = explained_indices[cn]
    sv   = shap_values[ci][ti]
    base = exp_vals[ci]
    top_idx2   = np.argsort(np.abs(sv))[::-1][:TOP_WF]
    top_vals2  = sv[top_idx2]
    top_nms2   = [feature_names[i] for i in top_idx2]
    cumvals    = np.concatenate([[base], base + np.cumsum(top_vals2)])
    bar_cols   = ['#1976D2' if v > 0 else '#E53935' for v in top_vals2]
    ax.barh(range(TOP_WF), top_vals2, left=cumvals[:-1],
            color=bar_cols, edgecolor='white', lw=0.5, height=0.6)
    ax.axvline(base, color='gray', lw=1, linestyle='--',
               label=f'Base={base:.3f}')
    ax.axvline(cumvals[-1], color='black', lw=1.5,
               label=f'f(x)={cumvals[-1]:.3f}')
    ax.set_yticks(range(TOP_WF))
    ax.set_yticklabels(top_nms2, fontsize=9)
    ax.set_xlabel('SHAP value contribution')
    ax.set_title(f'Class: {cn} | '
                 f'RF prob={rf.predict_proba(X_test_pca[ti:ti+1])[0][ci]:.3f}',
                 fontsize=10, fontweight='bold')
    ax.legend(fontsize=8, loc='lower right')
    ax.set_facecolor('#f9f9f9')
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '10_shap_waterfall.png'),
            dpi=150, bbox_inches='tight')
plt.close(); print("Saved: 10_shap_waterfall.png")

# ============================================================
# CELL 23 — PLOT 11: SHAP Decision Plot
# ============================================================
TOP_DEC = min(10, n_features)

fig, axes = plt.subplots(1, 3, figsize=(15, 7))
fig.suptitle('SHAP Decision Plots — All Test Samples\n(each line = one sample)',
             fontsize=13, fontweight='bold')
for ax, ci, cn in zip(axes, range(num_classes), class_names):
    sv    = shap_values[ci]
    base  = exp_vals[ci]
    top10 = np.argsort(np.abs(sv).mean(axis=0))[::-1][:TOP_DEC]
    sv10  = sv[:, top10]
    cum   = base + np.cumsum(sv10[:, ::-1], axis=1)
    yp    = np.arange(TOP_DEC, 0, -1)
    for i in range(len(sv10)):
        col   = '#1976D2' if y_test[i] == ci else '#cccccc'
        alpha = 0.7       if y_test[i] == ci else 0.12
        ax.plot(cum[i], yp, color=col, alpha=alpha, lw=0.6)
    ax.axvline(base, color='gray', lw=1, linestyle='--')
    ax.set_yticks(yp)
    ax.set_yticklabels([feature_names[i] for i in top10[::-1]], fontsize=8)
    ax.set_xlabel('Cumulative SHAP output')
    ax.set_title(f'Class: {cn}\n(blue = true class)',
                 fontsize=10, fontweight='bold')
    ax.set_facecolor('#f9f9f9')
    ax.grid(axis='x', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '11_shap_decision.png'),
            dpi=150, bbox_inches='tight')
plt.close(); print("Saved: 11_shap_decision.png")

# ============================================================
# CELL 24 — PLOT 12: SHAP Heatmap per class
# ============================================================
TOP_HM = min(15, n_features)

fig, axes = plt.subplots(1, 3, figsize=(15, 6))
fig.suptitle('SHAP Heatmaps — Feature Impact per Sample\n'
             '(sorted by predicted probability)',
             fontsize=13, fontweight='bold')
for ax, ci, cn in zip(axes, range(num_classes), class_names):
    sv    = shap_values[ci]
    top15 = np.argsort(np.abs(sv).mean(axis=0))[::-1][:TOP_HM]
    sv15  = sv[:, top15][np.argsort(rf_probs[:, ci])]
    im = ax.imshow(sv15.T, aspect='auto', cmap='RdBu_r',
                   vmin=-np.abs(sv15).max(), vmax=np.abs(sv15).max())
    ax.set_yticks(range(TOP_HM))
    ax.set_yticklabels([feature_names[i] for i in top15], fontsize=8)
    ax.set_xlabel('Test samples (sorted by P(class))')
    ax.set_title(f'Class: {cn}', fontsize=11, fontweight='bold')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='SHAP value')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '12_shap_heatmap.png'),
            dpi=150, bbox_inches='tight')
plt.close(); print("Saved: 12_shap_heatmap.png")

# ============================================================
# CELL 25 — PLOT 13: LIME vs SHAP Agreement pie charts
# ============================================================
TOP_AGR = min(10, n_features)

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
fig.suptitle('LIME vs SHAP — Top Feature Agreement per Class',
             fontsize=13, fontweight='bold')
for ax, cn in zip(axes, class_names):
    ci = class_names.index(cn)
    ti = explained_indices[cn]
    el = lime_rf_exp[cn].as_list(label=ci)
    lime_feats = set()
    for fname, _ in sorted(el, key=lambda x: abs(x[1]), reverse=True)[:TOP_AGR]:
        for fn in feature_names:
            if fn in fname:
                lime_feats.add(fn); break
    shap_top  = set(
        feature_names[i]
        for i in np.argsort(np.abs(shap_values[ci][ti]))[::-1][:TOP_AGR]
    )
    overlap   = lime_feats & shap_top
    lime_only = lime_feats - shap_top
    shap_only = shap_top   - lime_feats
    sizes = [len(overlap), len(lime_only), len(shap_only)]
    if sum(sizes) == 0: sizes = [1, 1, 1]
    ax.pie(sizes,
           labels=[f'Agreement\n({len(overlap)})',
                   f'LIME only\n({len(lime_only)})',
                   f'SHAP only\n({len(shap_only)})'],
           colors=['#388E3C', '#1976D2', '#F57C00'],
           autopct='%1.0f%%', explode=[0.05, 0, 0],
           startangle=90, textprops={'fontsize': 9})
    ax.set_title(f'Class: {cn}', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '13_lime_shap_agreement.png'),
            dpi=150, bbox_inches='tight')
plt.close(); print("Saved: 13_lime_shap_agreement.png")

# ============================================================
# CELL 26 — PLOT 14: t-SNE of SHAP Values
# ============================================================
print("\nt-SNE on SHAP space (max_iter=300)...")
shap_cat = np.concatenate([shap_values[c] for c in range(num_classes)], axis=1)
tsne     = TSNE(n_components=2,
                perplexity=min(30, len(shap_cat) - 1),
                random_state=RANDOM_SEED,
                max_iter=300, n_jobs=-1)
shap_2d  = tsne.fit_transform(shap_cat)
print("t-SNE done.")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('t-SNE of SHAP Value Space', fontsize=13, fontweight='bold')
colors_cls  = ['#1976D2', '#388E3C', '#E53935']
markers_cls = ['o', 's', '^']

ax = axes[0]
for ci, (cn, col, mk) in enumerate(zip(class_names, colors_cls, markers_cls)):
    mask = y_test == ci
    ax.scatter(shap_2d[mask, 0], shap_2d[mask, 1],
               c=col, marker=mk, s=35, alpha=0.75, label=cn,
               edgecolors='white', linewidths=0.4)
ax.set_title('Coloured by True Class', fontsize=11, fontweight='bold')
ax.legend(fontsize=10); ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
ax.set_facecolor('#f9f9f9'); ax.grid(alpha=0.25, linestyle='--')

ax = axes[1]
cm_mask = rf_preds == y_test
ax.scatter(shap_2d[cm_mask, 0],  shap_2d[cm_mask, 1],
           c='#388E3C', s=30, alpha=0.7, label='Correct',
           edgecolors='white', linewidths=0.4)
ax.scatter(shap_2d[~cm_mask, 0], shap_2d[~cm_mask, 1],
           c='#E53935', s=55, alpha=0.9, marker='X',
           label='Misclassified', edgecolors='white', linewidths=0.5)
ax.set_title('Coloured by Correctness', fontsize=11, fontweight='bold')
ax.legend(fontsize=10); ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
ax.set_facecolor('#f9f9f9'); ax.grid(alpha=0.25, linestyle='--')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '14_tsne_shap.png'),
            dpi=150, bbox_inches='tight')
plt.close(); print("Saved: 14_tsne_shap.png")

# ============================================================
# CELL 27 — PLOT 15: Final Results Table
# ============================================================
fig, ax = plt.subplots(figsize=(13, 4))
ax.axis('off')
col_labels = ['Model', 'Top-1', 'Top-2', 'Top-3',
              'Macro F1', 'AUC-ROC', 'XAI Method']
row_data = [
    ['Random Forest',
     f'{rf_top1:.4f}', f'{rf_top2:.4f}', f'{rf_top3:.4f}',
     f'{rf_f1:.4f}',   f'{rf_auc:.4f}',  'TreeSHAP + LIME'],
    ['SVM (RBF)',
     f'{svm_top1:.4f}', f'{svm_top2:.4f}', f'{svm_top3:.4f}',
     f'{svm_f1:.4f}',   f'{svm_auc:.4f}',  'KernelSHAP + LIME'],
]
tbl = ax.table(cellText=row_data, colLabels=col_labels,
               cellLoc='center', loc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.3, 2.5)
for j in range(len(col_labels)):
    tbl[0, j].set_facecolor('#1976D2')
    tbl[0, j].set_text_props(color='white', fontweight='bold')
for i in range(1, 3):
    for j in range(len(col_labels)):
        tbl[i, j].set_facecolor('#EFF6FF' if i % 2 == 0 else 'white')
ax.set_title('Final Results — LIME & SHAP on AAV Dataset\n'
             'CSE M275 | Dr. Mohammad Shahadat Hossain',
             fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, '15_final_table.png'),
            dpi=150, bbox_inches='tight')
plt.close(); print("Saved: 15_final_table.png")

# ============================================================
# CELL 28 — Final Checklist
# ============================================================
plot_files = [
    '01_confusion_matrices.png',  '02_classifier_topk.png',
    '03_roc_curves.png',          '04_lime_rf_features.png',
    '05_lime_rf_vs_svm.png',      '06_lime_summary.png',
    '07_lime_stability.png',      '08_shap_global_importance.png',
    '09_shap_summary_dots.png',   '10_shap_waterfall.png',
    '11_shap_decision.png',       '12_shap_heatmap.png',
    '13_lime_shap_agreement.png', '14_tsne_shap.png',
    '15_final_table.png',
]
print("\n" + "="*55)
print("PLOT CHECKLIST")
print("="*55)
all_ok = True
for p in plot_files:
    exists = os.path.exists(os.path.join(PLOT_DIR, p))
    if not exists: all_ok = False
    print(f"  {'OK' if exists else 'MISSING'}  {p}")

if all_ok:
    print(f"\nAll 15 plots saved to:\n  {PLOT_DIR}")
    print("\nDownload as zip:")
    print("  import shutil")
    print("  shutil.make_archive('/content/xai_results', 'zip', PLOT_DIR)")
    print("  from google.colab import files")
    print("  files.download('/content/xai_results.zip')")
else:
    print("\nSome plots missing — check errors above.")


SHAP TreeExplainer on Random Forest
Efficiency check — max diff: 2.39e-15
  (for RF multiclass, values up to ~0.5 are expected — this is normal)
Saved: 08_shap_global_importance.png
Saved: 09_shap_summary_dots.png
Saved: 10_shap_waterfall.png
Saved: 11_shap_decision.png
Saved: 12_shap_heatmap.png
Saved: 13_lime_shap_agreement.png

t-SNE on SHAP space (max_iter=300)...
t-SNE done.
Saved: 14_tsne_shap.png
Saved: 15_final_table.png

PLOT CHECKLIST
  OK  01_confusion_matrices.png
  OK  02_classifier_topk.png
  OK  03_roc_curves.png
  OK  04_lime_rf_features.png
  OK  05_lime_rf_vs_svm.png
  OK  06_lime_summary.png
  OK  07_lime_stability.png
  OK  08_shap_global_importance.png
  OK  09_shap_summary_dots.png
  OK  10_shap_waterfall.png
  OK  11_shap_decision.png
  OK  12_shap_heatmap.png
  OK  13_lime_shap_agreement.png
  OK  14_tsne_shap.png
  OK  15_final_table.png

All 15 plots saved to:
  /content/drive/MyDrive/SSL_Checkpoints/xai_plots

Download as zip:
  import shutil
  shutil.make_a

In [2]:
!git config --global user.name "fsananna"
!git config --global user.email "sultanafarhana55@gmail.com"

!git init

!git add .

!git commit -m "first commit"

!git branch -M main

!git remote add origin https://github.com/fsananna/Lime_shap_aav.git

!git push -u origin main

Reinitialized existing Git repository in /content/.git/
On branch main
nothing to commit, working tree clean
error: remote origin already exists.
fatal: could not read Username for 'https://github.com': No such device or address
